# 🗓️ 06 · 数据平台与调度：DAG 编排 / 依赖 / 重跑

> 数据平台 = 调度（Airflow/DolphinScheduler）+ 血缘 + 质量 + 权限。
> 本节手写**任务 DAG、依赖调度、失败重试、血缘追踪**（设 mini 调度器）。

### 🎬 背景故事

**几十个任务环环相扣，谁先跑谁后跑？谁失败了怎么办？……**

数据平台上有几十个任务环环相扣：DWD 清洗必须等 ODS 同步完成，DWS 汇总必须等 DWD 完成。**谁先跑、谁后跑、谁失败要不要重试、失败了下游怎么办？** 这就是调度系统（Airflow/DolphinScheduler）要解决的。本节手写拓扑排序和失败重试模拟。

**🧭 本节逻辑主线**：任务依赖怎么表示（DAG）→ 拓扑排序找执行顺序 → 失败重试与依赖阻断 → 血缘追踪 → 调度面试速答。


## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | 数据平台有哪些组件？ | 调度/血缘/质量/权限 |
| 2 | 任务依赖怎么表达？ | DAG（有向无环图） |
| 3 | 失败怎么处理？ | 重试/告警/上游阻断 |
| 4 | 血缘怎么维护？ | 表级依赖记录 |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
import sqlite3
import os
rng = np.random.default_rng(42)


## 1️⃣ 数据平台全景（必背）

In [2]:
print('''数据平台三大块：
  1) 开发平台：SQL 编辑器/任务发布/版本管理（如 DataWorks/LH）；
  2) 调度系统：DAG 编排、定时、依赖、重跑（Airflow/DolphinScheduler）；
  3) 治理体系：血缘、质量（规则稽核）、元数据、权限审计。

面试常问："你们调度任务失败怎么处理？血缘有什么用？质量怎么保证？"
''')

数据平台三大块：
  1) 开发平台：SQL 编辑器/任务发布/版本管理（如 DataWorks/LH）；
  2) 调度系统：DAG 编排、定时、依赖、重跑（Airflow/DolphinScheduler）；
  3) 治理体系：血缘、质量（规则稽核）、元数据、权限审计。

面试常问："你们调度任务失败怎么处理？血缘有什么用？质量怎么保证？"



### 📐 公式推导：任务 DAG、拓扑排序与重试期望

**DAG 与调度**：任务 $i$ 依赖集合 $\text{dep}(i)$，可执行条件 = 所有依赖成功：

$$ \text{run}(i) \iff \forall\, j \in \text{dep}(i):\; \text{status}(j) = \text{成功} $$

**拓扑排序**（找合法执行序）：反复取出"入度为 0"的任务（依赖全部就绪）；若某任务永远入度非零 → 图有环，调度死锁。用队列实现的 Kahn 算法 $O(V+E)$。

**失败重试的期望成功概率**（每次失败独立、成功率 $p$、最多重试 $k$ 次）：

$$ P(\text{最终成功}) = 1 - (1-p)^{k+1}, \qquad E[\text{尝试次数}] = \frac{1}{p}\big(1-(1-p)^{k+1}\big) $$

**推导要点**：① $p=0.9, k=2$ 时最终成功率 $1-0.1^3 = 99.9\%$——重试把"大概率成功"放大到"几乎必成功"；② 但重试要**幂等**（同一数据跑两次结果一致），否则下游重复计数；③ 血缘 = DAG 的投影：表 $T_1$ 由表 $T_0$ 产出则 $T_0 \to T_1$，回溯血缘可定位"哪个上游挂了导致今天报表缺数"。

## 2️⃣ 构造任务 DAG（字典表示）

In [3]:
# 🖼️ 配图：调度 DAG——任务依赖一目了然
import networkx as nx
dag = nx.DiGraph()
tasks = ['ods_log', 'dwd_clean', 'dwd_user', 'dws_daily', 'ads_top', 'ads_report']
edges = [('ods_log', 'dwd_clean'), ('ods_log', 'dwd_user'), ('dwd_clean', 'dws_daily'),
         ('dwd_user', 'dws_daily'), ('dws_daily', 'ads_top'), ('dws_daily', 'ads_report')]
dag.add_nodes_from(tasks); dag.add_edges_from(edges)
plt.figure(figsize=(6.5, 3.8))
pos = {'ods_log': (0, 3), 'dwd_clean': (2, 4.2), 'dwd_user': (2, 1.8),
       'dws_daily': (4, 3), 'ads_top': (6, 4.2), 'ads_report': (6, 1.8)}
nx.draw(dag, pos, with_labels=True, node_color='#93c5fd', node_size=2600,
        font_size=9, font_color='#0f172a', arrowsize=16, arrowstyle='-|>', width=1.8)
plt.title('任务 DAG：下游依赖上游成功（调度=找可执行顺序）', fontsize=11)
plt.show()
print('➡️ 拓扑排序：ods_log → dwd_clean/dwd_user → dws_daily → ads_*')

➡️ 拓扑排序：ods_log → dwd_clean/dwd_user → dws_daily → ads_*


C:\Users\24260\AppData\Local\Temp\ipykernel_30156\4152699438.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# 任务 DAG：每个任务依赖前序任务的产出
tasks = {
    'ods_sync':   [],
    'dwd_clean':  ['ods_sync'],
    'dws_agg':    ['dwd_clean'],
    'ads_report': ['dws_agg'],
    'ads_feature': ['dws_agg'],
}
for t, deps in tasks.items():
    print(f'  {t:<12} 依赖: {deps or "无（源任务）"}')
print('\n➡️ DAG 保证：ads_report 和 ads_feature 都等 dws_agg 完成')

  ods_sync     依赖: 无（源任务）
  dwd_clean    依赖: ['ods_sync']
  dws_agg      依赖: ['dwd_clean']
  ads_report   依赖: ['dws_agg']
  ads_feature  依赖: ['dws_agg']

➡️ DAG 保证：ads_report 和 ads_feature 都等 dws_agg 完成


## 3️⃣ 拓扑排序：找出可执行顺序

In [5]:
from collections import deque

def topo_sort(deps):
    indeg = {t: len(d) for t, d in deps.items()}
    adj = {t: [] for t in deps}
    for t, ds in deps.items():
        for d in ds:
            adj[d].append(t)
    q = deque([t for t, i in indeg.items() if i == 0])
    order = []
    while q:
        cur = q.popleft(); order.append(cur)
        for nxt in adj[cur]:
            indeg[nxt] -= 1
            if indeg[nxt] == 0:
                q.append(nxt)
    return order, len(order) == len(deps)

order, ok = topo_sort(tasks)
print('拓扑顺序:', order)
print('DAG 无环:', ok)
assert ok and order[0] == 'ods_sync'
print('✅ DAG 合法，调度顺序确定')

拓扑顺序: ['ods_sync', 'dwd_clean', 'dws_agg', 'ads_report', 'ads_feature']
DAG 无环: True
✅ DAG 合法，调度顺序确定


## 4️⃣ 失败重试与依赖阻断（模拟调度器）

In [6]:
# 模拟：dwd_clean 第一次失败，重试 2 次成功；观察下游被阻断
def run_task(name, fail_first, state):
    if fail_first and state.get(name, 0) == 0:
        state[name] = state.get(name, 0) + 1
        return False
    state[name] = state.get(name, 0) + 1
    return True

state = {}
plan = ['ods_sync', 'dwd_clean', 'dws_agg', 'ads_report']
fails = {'dwd_clean': 1}   # 第一次失败
log = []
for t in plan:
    attempt = 0
    success = False
    while attempt < 3 and not success:
        attempt += 1
        success = run_task(t, fails.get(t, 0) >= attempt, state)
        log.append((t, attempt, success))
    if not success:
        print(f'❌ {t} 重试 3 次仍失败 -> 阻断下游')
        break
    print(f'✅ {t} 第 {attempt} 次成功')
print('\n➡️ 调度策略：指数退避重试 + 依赖失败阻断下游 + 告警通知')

✅ ods_sync 第 1 次成功
✅ dwd_clean 第 2 次成功
✅ dws_agg 第 1 次成功
✅ ads_report 第 1 次成功

➡️ 调度策略：指数退避重试 + 依赖失败阻断下游 + 告警通知


## 5️⃣ 血缘追踪（表级依赖）

In [7]:
# 血缘：记录表与表之间的来源关系
lineage = [
    ('ods_orders', 'dwd_orders'),
    ('ods_users', 'dwd_orders'),
    ('dwd_orders', 'dws_daily_orders'),
    ('dws_daily_orders', 'ads_gmv_report'),
    ('dws_daily_orders', 'ads_user_feature'),
]
print('血缘（上游表 -> 下游表）:')
for up, down in lineage:
    print(f'  {up} -> {down}')
print('''
  血缘的用途：
  1) 影响分析：改一张 ODS 表，下游哪些报表/特征会变；
  2) 故障排查：报表数据异常时快速定位源头；
  3) 成本治理：找无下游的"孤儿表"下线省钱。
''')

血缘（上游表 -> 下游表）:
  ods_orders -> dwd_orders
  ods_users -> dwd_orders
  dwd_orders -> dws_daily_orders
  dws_daily_orders -> ads_gmv_report
  dws_daily_orders -> ads_user_feature

  血缘的用途：
  1) 影响分析：改一张 ODS 表，下游哪些报表/特征会变；
  2) 故障排查：报表数据异常时快速定位源头；
  3) 成本治理：找无下游的"孤儿表"下线省钱。



## 6️⃣ 调度系统面试速答

In [8]:
print('''1) 调度频率怎么定？ -> 分层：小时级/天级按数据时间（T-1 等）；依赖更细的任务用分钟级；
2) 数据时间 vs 运行时间？-> 今天 01:00 跑昨天的数据（T-1）；数据时间=业务时间；
3) 天级任务凌晨跑不完怎么办？-> 提前开始时间/并行度/拆任务/增量；
4) 任务积压如何追？-> 提高并行度、跳过不必要重跑、优先级；
5) 平台差异：Airflow（Python 生态）vs DolphinScheduler（可视化）.
''')

1) 调度频率怎么定？ -> 分层：小时级/天级按数据时间（T-1 等）；依赖更细的任务用分钟级；
2) 数据时间 vs 运行时间？-> 今天 01:00 跑昨天的数据（T-1）；数据时间=业务时间；
3) 天级任务凌晨跑不完怎么办？-> 提前开始时间/并行度/拆任务/增量；
4) 任务积压如何追？-> 提高并行度、跳过不必要重跑、优先级；
5) 平台差异：Airflow（Python 生态）vs DolphinScheduler（可视化）.



## 前沿练习：向量索引返回了 k 个邻居，就代表检索对吗？

2026-11 的数据库会议投稿里，向量检索正在从"插件"变成"一等公民"：[Errpedia](https://arxiv.org/abs/2611.22474) 直接质疑"你的 exact k-NN 真的 exact 吗"，[Pelorus](https://arxiv.org/abs/2611.22159) 做双曲空间图索引，[HorizANN](https://arxiv.org/abs/2611.19077) 用 ghost edges 做弹性扩容，[SIMQ](https://arxiv.org/abs/2611.22079) 与 [UniGraph](https://arxiv.org/abs/2611.20758) 分别解决多向量查询和稠密-稀疏混合检索，[DiskANN 的 beam search 变体](https://arxiv.org/abs/2611.26609) 在研究搜索预算怎么花。[HNSW 在 embedding 模型换代时的迁移](https://arxiv.org/abs/2611.25416) 则提醒：索引与模型是绑定资产。

工程上最容易犯的错是只看"返回了 top-k"，不看 **recall@k**：量化会悄悄改变距离排序，IVF 的 nprobe 太小会漏掉整个簇。下面的练习用合成向量做一次召回审计——所有配置都返回 k 个结果，但正确率差很多。


In [ ]:
import math
import numpy as np
rng = np.random.default_rng(0)

n, d, k = 1500, 32, 10
centers = rng.standard_normal((20, d))
X = centers[rng.integers(0, 20, n)] + 0.35 * rng.standard_normal((n, d))
q = X[rng.integers(0, n)] + 0.05 * rng.standard_normal(d)


def topk(dist, k):
    return np.argsort(dist)[:k]


def recall(gt, pred):
    return len(set(gt.tolist()) & set(pred.tolist())) / k


gt = topk(np.linalg.norm(X - q, axis=1), k)

# --- 1) 乘积量化（PQ）：每 8 维一个子空间，16 个码字 ----------
sub, ncode = d // 4, 16
codes, cents = [], []
for s in range(4):
    Xs = X[:, s * sub:(s + 1) * sub]
    C = Xs[rng.choice(n, ncode, replace=False)].copy()
    for _ in range(8):
        assign = np.argmin(((Xs[:, None, :] - C[None]) ** 2).sum(-1), axis=1)
        for c in range(ncode):
            if (assign == c).any():
                C[c] = Xs[assign == c].mean(0)
    codes.append(assign)
    cents.append(C)

approx = np.zeros(n)
for s in range(4):
    approx += ((cents[s][codes[s]] - q[None, s * sub:(s + 1) * sub]) ** 2).sum(1)
recall_pq = recall(gt, topk(approx, k))

# --- 2) IVF：粗量化 + nprobe 扫描簇 ----------
nc = 16
coarse = X[rng.choice(n, nc, replace=False)].copy()
assign = np.argmin(((X[:, None, :] - coarse[None]) ** 2).sum(-1), axis=1)
qcell = int(np.argmin(((coarse - q[None]) ** 2).sum(1)))
order = np.argsort(((coarse - q[None]) ** 2).sum(1))

def ivf_recall(nprobe):
    cells = order[:nprobe]
    cand = np.where(np.isin(assign, cells))[0]
    dist = np.linalg.norm(X[cand] - q, axis=1)
    return recall(gt, cand[topk(dist, k)])

recalls = [ivf_recall(p) for p in (1, 2, 4, nc)]
print(f"PQ 召回@10 = {recall_pq:.2f}（仍然返回 10 个结果，只是有错）")
print("IVF 召回@10 随 nprobe =", [round(r, 2) for r in recalls])

assert recall_pq < 1.0, "量化距离会让排序改变，召回不可能无损"
assert recalls[-1] == 1.0, "扫描全部簇等价于暴力检索，召回应为 1"
assert recalls[0] <= recalls[1] <= recalls[2] <= recalls[3], "nprobe 越大召回应不减"
assert recalls[0] < 1.0, "nprobe=1 往往会漏掉真正最近的簇"
print("结论：所有配置都返回 top-k，但 recall@k 才是验收指标")


## 📝 自测清单

- [ ] 能画出数据平台三块全景
- [ ] 能写 DAG 并表示任务依赖
- [ ] 能实现拓扑排序
- [ ] 能讲失败重试与依赖阻断策略
- [ ] 能说出血缘三大用途